# SSS Debris Detection — Stage 2 (Fixed)

## Problem
- Only 153 unique debris images (460 with augmentation)
- Tiny objects: 32x12 pixels in 512x512 images
- Standard training overfits immediately

## Solution
1. **Copy-paste augmentation** — Paste debris onto random backgrounds
2. **Freeze backbone** — Keep pretrained features, only train detection head
3. **Lower learning rate** — Stable fine-tuning
4. **Evaluate at conf=0.05** — Find more debris detections

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 1: Setup (single clone, no nesting)
# ═══════════════════════════════════════════════════════════
!pip install ultralytics pandas matplotlib -q

# Clean clone (remove any existing)
!rm -rf sonarvision
!git clone https://github.com/Dinoman67/sonarvision.git
%cd sonarvision

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 2: Upload and balance dataset
# ═══════════════════════════════════════════════════════════
from google.colab import files
uploaded = files.upload()  # Upload e5.zip
!unzip -q e5.zip -d /content/

# Balance dataset
!python scripts/balance_e5_dataset.py \
    --e5 /content/e5 \
    --output /content/e5_balanced \
    --ratio 3

DATA = '/content/e5_balanced/data.yaml'
print(f'\nDataset ready: {DATA}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 3: Load modules
# ═══════════════════════════════════════════════════════════
from ultralytics import YOLO
from models.sss_custom_modules import PConv, FasterBlock, FastC2f, GhostConv, SEBlock
from models.build_sss_models import build_ss_yolo, build_yolov8_esi_full, C2fWithSE
from ultralytics.models.yolo.detect.train import DetectionTrainer
print('✓ Modules loaded')

## ═══════════════════════════════════════════════════════════
## TRAINING: YOLOv8-ESI with Frozen Backbone
## ═══════════════════════════════════════════════════════════

Key changes:
- **Freeze first 10 layers** (backbone) — keeps pretrained features
- **copy_paste=0.5** — paste debris onto backgrounds during training
- **lr0=0.001** — lower LR for stable fine-tuning
- **epochs=100** with patience=30

In [ ]:
# ═══════════════════════════════════════════════════════════
# Train YOLOv8-ESI (frozen backbone + copy-paste)
# ═══════════════════════════════════════════════════════════
print('='*60)
print('Training YOLOv8-ESI (frozen backbone + copy-paste)')
print('='*60)

# Build fresh YOLOv8-ESI
esi_model = build_yolov8_esi_full()

_orig = DetectionTrainer.get_model
def _patched_esi(self, cfg=None, weights=None, verbose=True):
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.utils import RANK
    model = self.set_model_names_for_load(
        DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'],
                       verbose=verbose and RANK == -1)
    )
    model.model = esi_model.model
    model.nc = 1
    model.names = {0: 'marine_debris'}
    try:
        model.load(weights)
    except Exception:
        pass
    return model
DetectionTrainer.get_model = _patched_esi

try:
    yolo_esi = YOLO('yolov8n.pt')
    yolo_esi.train(
        data=DATA,
        epochs=100,
        imgsz=256,
        batch=16,
        patience=30,
        lr0=0.001,         # Lower LR for frozen backbone
        lrf=0.01,
        warmup_epochs=3,
        freeze=10,         # Freeze first 10 layers (backbone)
        copy_paste=0.5,    # Paste debris onto backgrounds
        mosaic=0.0,
        mixup=0.0,
        fliplr=0.0,
        flipud=0.0,
        degrees=0.0,
        translate=0.05,
        scale=0.2,
        name='yolov8_esi_frozen',
        project='/content/runs',
        exist_ok=True,
        plots=True,
    )

    # Evaluate with best weights
    print('\n--- Evaluating best model ---')
    best = YOLO('/content/runs/yolov8_esi_frozen/weights/best.pt')
    
    # Sweep confidence thresholds
    print('\nConfidence sweep (val set):')
    print(f'{"Conf":>6} {"P":>8} {"R":>8} {"mAP50":>8} {"F1":>8}')
    print('-' * 42)
    
    best_f1, best_conf = 0, 0.05
    for c in [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
        r = best.val(data=DATA, imgsz=256, conf=c, verbose=False)
        p, rv = r.box.mp, r.box.mr
        f1 = 2*p*rv / max(p+rv, 1e-8)
        marker = ' ←' if f1 > best_f1 else ''
        print(f'{c:>6.2f} {p:>8.4f} {rv:>8.4f} {r.box.map50:>8.4f} {f1:>8.4f}{marker}')
        if f1 > best_f1:
            best_f1, best_conf = f1, c
    
    print(f'\nOptimal conf: {best_conf}')
    
    # Final test evaluation
    print('\n--- TEST SET ---')
    test_r = best.val(data=DATA, imgsz=256, conf=best_conf, split='test')
    p, rv = test_r.box.mp, test_r.box.mr
    f1 = 2*p*rv / max(p+rv, 1e-8)
    
    print(f'\n{"="*60}')
    print(f'FINAL RESULTS: YOLOv8-ESI (frozen backbone)')
    print(f'{"="*60}')
    print(f'  Test mAP50:  {test_r.box.map50:.4f}')
    print(f'  Test P:      {p:.4f}')
    print(f'  Test R:      {rv:.4f}')
    print(f'  Test F1:     {f1:.4f}')
    print(f'  Best conf:   {best_conf}')
    
except Exception as e:
    print(f'✗ Failed: {e}')
    import traceback; traceback.print_exc()
finally:
    DetectionTrainer.get_model = _orig

In [ ]:
# ═══════════════════════════════════════════════════════════
# Export best model
# ═══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('EXPORT')
print('='*60)

best.export(format='onnx', imgsz=256)
print('\n✓ Model exported to ONNX')

# Download
from google.colab import files
files.download('/content/runs/yolov8_esi_frozen/weights/best.pt')
files.download('/content/runs/yolov8_esi_frozen/weights/best.onnx')